# Demo - Escalation Driven by a Real Test Failure
**Day 1 - Session 1, Topic 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-escalation-from-failure.ipynb)

**Goal:** Break the email renderer for real, capture the actual unittest failure, and derive the routing decision from that captured evidence.

Nothing about the decision is pre-written. The notebook edits `channels/email.py` in a throwaway copy so untrusted input stops being escaped, runs the real security test, and reads the genuine failure text. Repair the file, re-run the same rule, and the answer changes.

> Edits a temporary copy only. No API key and no network needed.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [1]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

Course root: /Users/kangs/code/github/agent-orchestration-companion
Git: git version 2.50.1 (Apple Git-155)


## 2. Confirm the approved baseline passes

Before breaking anything, prove the tests are green so the failure is unambiguous.


In [2]:
import re
from pathlib import Path

from demo_support import assert_true, heading, run_tests, sandbox, show_evidence

TARGET = Path("channels") / "email.py"
SECURITY_TEST = "tests.test_security.ChannelSecurityTest.test_email_escapes_untrusted_html"
FOCUSED_TEST = "tests.test_email"

sandbox_context = sandbox(with_git=False)
WORK, _ = sandbox_context.__enter__()
ORIGINAL = (WORK / TARGET).read_text()

heading("Baseline: the approved implementation")
security_ok, _ = run_tests(SECURITY_TEST, WORK)
focused_ok, _ = run_tests(FOCUSED_TEST, WORK)
show_evidence("security assertion", "passed" if security_ok else "FAILED")
show_evidence("focused email tests", "passed" if focused_ok else "FAILED")


Baseline: the approved implementation
-------------------------------------
  security assertion         passed
  focused email tests        passed


## 3. Inject a real regression and capture the genuine failure

Remove `escape()` from the untrusted field, then read what unittest actually reports.


In [3]:
REGRESSION = ('escape(str(event["customer_name"]))', 'str(event["customer_name"])')

path = WORK / TARGET
path.write_text(ORIGINAL.replace(*REGRESSION))

heading("Inject a real regression")
show_evidence("file edited", str(TARGET))
show_evidence("change made", "removed escape() from the untrusted field")

security_ok, security_output = run_tests(SECURITY_TEST, WORK)
focused_ok, _ = run_tests(FOCUSED_TEST, WORK)
FAILURES = re.findall(r"^(?:FAIL|ERROR): (\S+)", security_output, flags=re.MULTILINE)
assertion_line = next(
    (line.strip() for line in security_output.splitlines() if line.startswith("AssertionError")),
    "(no assertion line found)",
)

heading("Run the real tests and capture the genuine failure")
show_evidence("security assertion", "passed" if security_ok else "FAILED")
show_evidence("focused email tests", "passed" if focused_ok else "FAILED")
show_evidence("unittest reported", ", ".join(FAILURES) or "(none)")
print(f"\n  actual assertion text:\n  {assertion_line}")


Inject a real regression
------------------------
  file edited                channels/email.py
  change made                removed escape() from the untrusted field

Run the real tests and capture the genuine failure
--------------------------------------------------
  security assertion         FAILED
  focused email tests        passed
  unittest reported          test_email_escapes_untrusted_html

  actual assertion text:
  AssertionError: '<script>' unexpectedly found in "Hello <script>alert('x')</script>, your order shipped via Swift\nShip. Track it at https://track.example/A100"


## 4. Measure the blast radius from real imports

Count the modules that genuinely reference the changed module.


In [4]:
def blast_radius(root, module_relative):
    dotted = str(module_relative.with_suffix("")).replace("/", ".")
    importers = []
    for candidate in root.rglob("*.py"):
        if "__pycache__" in candidate.parts or candidate == root / module_relative:
            continue
        if dotted in candidate.read_text():
            importers.append(str(candidate.relative_to(root)))
    return sorted(importers)


IMPORTERS = blast_radius(WORK, TARGET)
heading("Blast radius measured from the repository")
show_evidence("modules importing it", ", ".join(IMPORTERS) or "(none)")
show_evidence("count", len(IMPORTERS))


Blast radius measured from the repository
-----------------------------------------
  modules importing it       router.py, tests/test_email.py, tests/test_security.py
  count                      3


## 5. Apply the routing rule to the measured evidence

One rule reads the captured facts. A failing security assertion stops for a human.


In [5]:
def route(security_failed, importer_count, focused_failed):
    if security_failed:
        return ("STOP - human review",
                "A security assertion failed, so a stronger model is not the remedy.")
    if importer_count >= 1 and focused_failed:
        return ("ESCALATE - stronger model, high effort",
                "The failure reaches other modules, so widen reasoning before retrying.")
    if focused_failed:
        return ("KEEP - same model, narrowed scope",
                "The failure is local and reproducible, so inspect the smallest file set.")
    return ("KEEP - no change needed", "Checks pass, so no routing change is justified.")


heading("Routing decision derived from that evidence")
DECISION, reason = route(not security_ok, len(IMPORTERS), not focused_ok)
show_evidence("decision", DECISION)
show_evidence("because", reason)


Routing decision derived from that evidence
-------------------------------------------
  decision                   STOP - human review
  because                    A security assertion failed, so a stronger model is not the remedy.


## 6. Repair the file and re-apply the identical rule

The rule does not change. Only the evidence does, and the decision follows it.


In [6]:
path.write_text(ORIGINAL)
security_ok, _ = run_tests(SECURITY_TEST, WORK)
focused_ok, _ = run_tests(FOCUSED_TEST, WORK)
REPAIRED, repaired_reason = route(not security_ok, len(IMPORTERS), not focused_ok)

heading("After repair")
show_evidence("security assertion", "passed" if security_ok else "FAILED")
show_evidence("decision", REPAIRED)
show_evidence("because", repaired_reason)

heading("Evidence checks")
assert_true(FAILURES, "unittest produced real failure identifiers")
assert_true("STOP" in DECISION, "a failing security assertion stopped execution for a human")
assert_true(DECISION != REPAIRED, "the same rule changed answer when the evidence changed")
assert_true("router.py" in " ".join(IMPORTERS), "blast radius was measured from real imports")

sandbox_context.__exit__(None, None, None)
print("\nTakeaway: Escalate on captured evidence, not on repeated attempts,")
print("and stop for a human when the evidence is a security failure.")


After repair
------------
  security assertion         passed
  decision                   KEEP - no change needed
  because                    Checks pass, so no routing change is justified.

Evidence checks
---------------
  [verified] unittest produced real failure identifiers
  [verified] a failing security assertion stopped execution for a human
  [verified] the same rule changed answer when the evidence changed
  [verified] blast radius was measured from real imports

Takeaway: Escalate on captured evidence, not on repeated attempts,
and stop for a human when the evidence is a security failure.


### Expected output

- Baseline: both the security assertion and the focused email tests pass.
- After the regression the security assertion **FAILS** and unittest names
  `test_email_escapes_untrusted_html`.
- The real assertion text shows the `<script>alert('x')</script>` payload
  reaching the rendered body.
- Blast radius lists `router.py` plus the two test modules.
- Decision: `STOP - human review`. After repair the same rule returns
  `KEEP - no change needed`.
- Four `[verified]` lines, then the takeaway.
